In [1]:
from pathlib import Path
import os
import shutil


In [2]:
from IPython import get_ipython
from IPython.core.magic import register_cell_magic

ipython = get_ipython()


@register_cell_magic
def pybash(line, cell):
    cell_replaced = eval("f" + repr(cell))
    # print("Evaluating:\n{}\n-----------".format(cell_replaced))
    ipython.run_cell_magic('bash', '', cell_replaced)

In [3]:
project_dir = Path(os.getcwd()).parent.parent
install_dir = project_dir / "install"
log_dir = project_dir / "logs" / "dlio"
data_dir = Path("/p/lustre5/haridev/dlio_demo")
output_dir = project_dir / "output" / "dlio"
print("Directories created:")
for name, path in [("Install Directory", install_dir), 
                   ("Log Directory", log_dir), 
                   ("Data Directory", data_dir), 
                   ("Output Directory", output_dir)]:
    print(f"{name}: {path}")

Directories created:
Install Directory: /usr/WS2/haridev/dftracer-demo/install
Log Directory: /usr/WS2/haridev/dftracer-demo/logs/dlio
Data Directory: /p/lustre5/haridev/dlio_demo
Output Directory: /usr/WS2/haridev/dftracer-demo/output/dlio


In [4]:

for dir_path in [log_dir, data_dir, output_dir]:
    if dir_path.exists():
        for item in dir_path.iterdir():
            if item.is_file():
                item.unlink()
            elif item.is_dir():
                shutil.rmtree(item)
    dir_path.mkdir(parents=True, exist_ok=True)
print("Cleaned and created fresh folders for log, data, and output.")

Cleaned and created fresh folders for log, data, and output.


In [5]:
import os

import importlib.util

spec = importlib.util.find_spec("dftracer")
if spec and spec.origin:
    print("dftracer module path:", spec.origin)
    dftracer_folder = os.path.dirname(spec.origin)
    print("dftracer folder:", dftracer_folder)
else:
    print("dftracer module not found.")

dftracer module path: /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dftracer/__init__.py
dftracer folder: /usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dftracer


[Documentation](https://dftracer.readthedocs.io/en/latest/api.html)

In [ ]:
%%pybash
# DFTracer environment variables:
echo "Configuring DFTracer"

# DFTRACER_INC_METADATA: Include or exclude metadata (default 0)
export DFTRACER_INC_METADATA=1

# DFTRACER_ENABLE: Enable or Disable DFTracer (default 0).
export DFTRACER_ENABLE=1

echo "Activating environment"
source {project_dir}/demo/dlio/setup_env.sh {install_dir} 2> /dev/null


echo "Running DLIO with DFTracer"
flux run -n 2 -o fastload -q pbatch {install_dir}/bin/dlio_benchmark workload=unet3d_a100 ++workload.workflow.generate_data=True hydra.run.dir={output_dir}/unet3d_a100/ ++workload.output.folder={output_dir}/unet3d_a100/ ++workload.output.folder={output_dir}/unet3d_a100/ ++workload.dataset.num_files_train=32 ++workload.dataset.record_length_bytes=1048576 ++workload.dataset.data_folder={data_dir}/unet3d_a100/data ++workload.checkpoint.checkpoint_folder={data_dir}/unet3d_a100/checkpoint ++workload.train.epochs=1 > {output_dir}/log.txt 2> {output_dir}/error.txt
echo "Finished running DLIO with DFTracer"

Configuring DFTracer


Running DLIO with DFTracer


In [14]:
import glob

pfw_files = glob.glob(str(output_dir/ "unet3d_a100" / "trace-*-of-*.pfw.gz"))
if pfw_files:
    print("Found .pfw.gz files:")
    for f in pfw_files:
        print(f)
else:
    print("No .pfw.gz files found in", log_dir)
# Clean other files in output_dir/"unet3d_a100" apart from pfw_files
unet3d_dir = output_dir / "unet3d_a100"
for file in unet3d_dir.iterdir():
    if str(file) not in pfw_files:
        if file.is_file():
            file.unlink()
        elif file.is_dir():
            shutil.rmtree(file)
print("Cleaned non-.pfw.gz files in", unet3d_dir)



Found .pfw.gz files:
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-0-of-2.pfw.gz
/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/trace-1-of-2.pfw.gz
Cleaned non-.pfw.gz files in /usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100


In [15]:
!gzip -dc {output_dir}/unet3d_a100/*.pfw.gz | (head -n 10; echo "..."; tail -n 5)

[
{"id":1,"name":"HH","cat":"dftracer","pid":558623,"tid":558623,"ph":"M","args":{"hhash":"515b3a6191ad3cde","name":"tuolumne1744","value":"515b3a6191ad3cde"}}
{"id":2,"name":"thread_name","cat":"dftracer","pid":558623,"tid":558623,"ph":"M","args":{"hhash":"515b3a6191ad3cde","name":"558623","value":"thread_name"}}
{"id":3,"name":"SH","cat":"dftracer","pid":558623,"tid":558623,"ph":"M","args":{"hhash":"515b3a6191ad3cde","name":"/usr/WS2/haridev/dftracer-demo/install/bin/python;/usr/WS2/haridev/dftracer-demo/install/bin/dlio_benchmark;workload=unet3d_a100;++workload.workflow.generate_data=True;hydra.run.dir=/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/;++workload.output.folder=/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/;++workload.output.folder=/usr/WS2/haridev/dftracer-demo/output/dlio/unet3d_a100/;++workload.dataset.num_files_train=32;++workload.dataset.record_length_bytes=1048576;++workload.dataset.data_folder=/p/lustre5/haridev/dlio_demo/unet3d_a100/data;++workl

In [20]:
from dfanalyzer import init_with_hydra
dfa = init_with_hydra(
    hydra_overrides=[
        f"trace_path={output_dir}/unet3d_a100/",
        f"analyzer/preset=dlio",
        
    ]
)

/usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/distributed/node.py:187: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 39253 instead
  warnings.warn(


In [17]:
dfa.client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:43137/status,
Dashboard: http://127.0.0.1:43137/status,Workers: 12
Total threads: 96,Total memory: 0 B
Status: running,Using processes: True
Comm: tcp://127.0.0.1:41427,Workers: 12
Dashboard: http://127.0.0.1:43137/status,Total threads: 96
Started: Just now,Total memory: 0 B
Comm: tcp://127.0.0.1:33401,Total threads: 8
Dashboard: http://127.0.0.1:41765/status,Memory: 0 B
Nanny: tcp://127.0.0.1:38377,


In [21]:
res = dfa.analyze_trace()
dfa.output.handle_result(res)

/usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dask_expr/_collection.py:5063: FutureWarning: from_legacy_dataframe is deprecated and will be removed in a future release. The legacy implementation as a whole is deprecated and will be removed, making this method unnecessary.
  warnings.warn(
/usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dfanalyzer/output.py:112: RuntimeWarning: invalid value encountered in scalar divide
  ops=float('nan') if pd.isna(time) else float(count / time),
/usr/workspace/haridev/dftracer-demo/install/lib/python3.11/site-packages/dfanalyzer/output.py:113: RuntimeWarning: invalid value encountered in scalar divide
  bandwidth=float('nan') if pd.isna(time) or pd.isna(size) else float(size / time),


                                                Time Period Summary                                                
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ Metric                                                                     ┃ Unit            ┃            Value ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ Job Time                                                                   │ seconds         │           31.115 │
│ Total Count                                                                │ count           │            1,421 │
│ Total Files                                                                │ count           │                0 │
│ Total Nodes                                                                │ count           │                0 │
│ Total Processes                                                            │ count           │                2 │
│ App Count                                                                  │ count           │                2 │
│ Training Count                                                             │ count           │                2 │
│ Compute Count                                                              │ count           │                4 │
│ Fetch Data Count                                                           │ count           │               34 │
│ Data Loader Fork Count                                                     │ count           │               16 │
│ Reader POSIX (Lustre) Count                                                │ count           │            1,343 │
│ Reader POSIX (Lustre) Size                                                 │ MB              │         7889.868 │
│ Reader POSIX (Lustre) Bandwidth                                            │ MB/s            │         1084.725 │
│ Reader POSIX (Lustre) Avg Transfer Size                                    │ MB              │            5.875 │
│ Checkpoint Count                                                           │ count           │                1 │
│ Checkpoint POSIX (Lustre) Count                                            │ count           │                3 │
│ Other POSIX Count                                                          │ count           │               16 │
└────────────────────────────────────────────────────────────────────────────┴─────────────────┴──────────────────┘
                                          Layer Breakdown (w/ overlap %)                                           
┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer                        ┃      Time (s) ┃           Ops ┃   Ops/sec ┃        Size (MB) ┃  Bandwidth (MB/s) ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ App                          │  9.859 (----) │      2 (----) │     0.203 │                - │                 - │
│ Training                     │  9.804 (----) │      2 (----) │     0.204 │                - │                 - │
│ Compute                      │  2.544 (----) │      4 (----) │     1.572 │                - │                 - │
│ Fetch Data                   │  7.025 (  0%) │     34 (  0%) │     4.840 │                - │                 - │
│ Data Loader Fork             │  0.107 (  0%) │     16 (  0%) │   148.846 │                - │                 - │
│ Reader POSIX (Lustre)        │  7.274 (  0%) │  1,343 (  0%) │   184.640 │  7889.868 (  0%) │          1084.725 │
│ Checkpoint                   │  0.060 (  0%) │      1 (  0%) │    16.562 │                - │                 - │
│ Checkpoint POSIX (Lustre)    │  0.002 (  0%) │      3 (  0%) │  1671.309 │         - (----) │             0.000 │
│ Other POSIX                  │  0.107 (  0%) │     16 